# Инференс: генерация Python-кода по задачам TinySMS

Лучший подтверждённый результат — **public pass@1 = 61,77% (593/960)**, checkpoint exp09.
Архитектура: encoder–decoder Transformer, 5+5 слоёв, d_model=512, SwiGLU d_ff=2048,
8 голов, Pre-LayerNorm и RoPE; 51 484 672 параметра при vocab=8192.

Веса предоставляются отдельно. На Kaggle подключите их через **Add Input**; в Colab
загрузите архив/каталог в `/content` или подключите Google Drive. Нужны
`config.yaml`, `tokenizer.json` и `model_best_greedy.pth` либо `model_best.pth`.
Ссылок и ключей доступа в notebook нет.

**Выберите одну NVIDIA T4 и включите Internet.** Если Kaggle предоставляет T4 ×2,
код использует только устройство 0. Запускайте ячейки по порядку или Save & Run All.
Тренировочные данные, Comet и ключи API не нужны. Ноутбук загружает только test,
использует greedy + KV cache, FP32, batch=16, max_new_tokens=1024.

Результат — `submission.jsonl` из 1919 строк и `inference_metrics.json`.
Генерация измеряется на полном test; ограничение задания — 180 секунд на одной T4.
Отдельно измеряется весь процесс inference.py, включая загрузку весов.
Установка окружения и подготовка вопросов выполняются до измерения.

## 1. Пути и установка

Измените **MODEL_SOURCE**: это путь к каталогу одного запуска либо ZIP с моделью.
Для ZIP с несколькими моделями сначала выберите и распакуйте нужный каталог.
`TEST_PARQUET=None` скачивает официальный тест закреплённой версии; для локального
файла укажите путь. Полные train-шарды не загружаются.

`REPO_REF` — ветка или тег при клонировании кода. После установки печатается фактический commit.
На Kaggle сохраняйте эту установочную ячейку внутри notebook для Save & Run All.

In [ ]:
print("Inference notebook: setup started", flush=True)

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path, PurePosixPath
from time import perf_counter
from zipfile import ZipFile

WORK_DIR = (
    Path("/kaggle/working") if Path("/kaggle/working").is_dir()
    else Path("/content") if Path("/content").is_dir()
    else Path.cwd()
)
MODEL_SOURCE = Path("/kaggle/input/YOUR_MODEL/saved/exp09_finetune_fp32_b32_30k")
TEST_PARQUET = None  # например, Path("/kaggle/input/YOUR_TEST/test.parquet")
REPO_REF = "main"
REPO_URL = "https://github.com/Mishaq23/hw2_nlp.git"

MODEL_SOURCE = MODEL_SOURCE.expanduser().resolve()
if TEST_PARQUET is not None:
    TEST_PARQUET = Path(TEST_PARQUET).expanduser().resolve()

# Ограничиваем процесс одной GPU до любого импорта torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"
if not MODEL_SOURCE.exists():
    raise FileNotFoundError(f"Укажите доступный MODEL_SOURCE: {MODEL_SOURCE}")

repo = WORK_DIR / "hw2_nlp"
if not repo.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(repo)],
        check=True,
    )
print("Code commit:", flush=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(repo / "requirements.txt")],
    check=True,
)
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
if not (repo / "src/configs/best_inference.yaml").is_file():
    raise FileNotFoundError("Нужна версия репозитория с best_inference.yaml")

# Проверка CUDA в отдельном процессе не оставляет CUDA-контекст в notebook.
subprocess.run(
    [sys.executable, "-u", "-c", "import torch; "
     "assert torch.cuda.is_available(), 'Включите GPU T4'; "
     "print('PyTorch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0), flush=True)"],
    check=True,
)


## 2. Модель

Используется best для greedy. Если облегчённого файла нет, загружается полный
`model_best.pth`. `checkpoint-last.pth` автоматически не выбирается.
Из ZIP извлекаются только нужные три файла; train и optimizer для инференса не нужны.

In [ ]:
if MODEL_SOURCE.is_dir():
    model_dir = MODEL_SOURCE
    checkpoint = next(
        (model_dir / name for name in ("model_best_greedy.pth", "model_best.pth")
         if (model_dir / name).is_file()),
        None,
    )
else:
    model_dir = WORK_DIR / "inference_model"
    model_dir.mkdir(parents=True, exist_ok=True)
    with ZipFile(MODEL_SOURCE) as archive:
        members = {PurePosixPath(name): name for name in archive.namelist()
                   if not name.endswith("/")}
        candidates = [
            member.parent for member in members
            if member.name == "config.yaml"
            and member.parent / "tokenizer.json" in members
            and any(member.parent / name in members
                    for name in ("model_best_greedy.pth", "model_best.pth"))
        ]
        if len(candidates) != 1:
            raise ValueError(f"В ZIP нужен ровно один каталог модели: {candidates}")
        prefix = candidates[0]
        weight_name = next(name for name in ("model_best_greedy.pth", "model_best.pth")
                           if prefix / name in members)
        for name in ("config.yaml", "tokenizer.json", weight_name):
            print("Extract:", name, flush=True)
            with archive.open(members[prefix / name]) as src, (model_dir / name).open("wb") as dst:
                shutil.copyfileobj(src, dst)
        checkpoint = model_dir / weight_name

for required in (model_dir / "config.yaml", model_dir / "tokenizer.json"):
    if not required.is_file():
        raise FileNotFoundError(required)
if checkpoint is None:
    raise FileNotFoundError("Нужен model_best_greedy.pth или model_best.pth")

print("Model directory:", model_dir, flush=True)
print("Checkpoint:", checkpoint.name, flush=True)
print("Checkpoint size, MiB:", round(checkpoint.stat().st_size / 2**20, 1))


## 3. Только тестовые вопросы

Токенайзер загружается из модели и не обучается заново. Тест кодируется без
усечения; исходный порядок фиксируется через sample_id=0…1918.
Ни разбиение train/val, ни подготовка миллионов обучающих примеров не запускаются.

In [ ]:
from datasets import load_dataset
from src.datasets import CodeTokenizer, DATASET_ID, DATASET_REVISION, tokenize_dataset

prepared = WORK_DIR / "inference_data"
prepared.mkdir(parents=True, exist_ok=True)
cache_dir = WORK_DIR / "inference_hf_cache"
if TEST_PARQUET is None:
    raw_test = load_dataset(
        DATASET_ID,
        revision=DATASET_REVISION,
        data_files={"test": "data/test/test.parquet"},
        split="test",
        cache_dir=str(cache_dir),
    )
else:
    raw_test = load_dataset(
        "parquet", data_files={"test": str(TEST_PARQUET)},
        split="test", cache_dir=str(cache_dir),
    )
if len(raw_test) != 1919:
    raise ValueError(f"Ожидалось 1919 вопросов, получено {len(raw_test)}")
questions = raw_test["question"]
if not all(isinstance(question, str) and question.strip() for question in questions):
    raise ValueError("Тест содержит пустые вопросы")

raw_test = raw_test.select_columns(["question"]).add_column("sample_id", list(range(1919)))
tokenizer = CodeTokenizer.from_file(model_dir / "tokenizer.json")
encoded_test = tokenize_dataset(raw_test, tokenizer, has_targets=False)
encoded_test.save_to_disk(str(prepared / "encoded/test"))
shutil.copy2(model_dir / "tokenizer.json", prepared / "tokenizer.json")
os.environ["TINYSMS_DATA_DIR"] = str(prepared)
print("Test examples:", len(encoded_test), "vocabulary:", tokenizer.vocab_size, flush=True)


## 4. Генерация на одной T4

`inference.py` читает архитектуру из сохранённого `config.yaml`.
Настройки сдачи закреплены в `best_inference`: greedy, max_new_tokens=1024,
batch=16, FP32. Обучение, beam search и компиляция здесь не запускаются.
Время `generation_seconds` включает полный проход DataLoader, генерацию,
декодирование, синтаксическую проверку и запись predictions; модель уже загружена.
`inference_process_seconds` дополнительно включает запуск процесса и загрузку.

In [ ]:
output = WORK_DIR / "inference_output"
started = perf_counter()
subprocess.run(
    [
        sys.executable, "-u", "inference.py", "-cn", "best_inference",
        f"inferencer.checkpoint_dir={model_dir}",
        f"inferencer.from_pretrained={checkpoint}",
        f"inferencer.save_path={output}",
        "inferencer.device=cuda",
    ],
    cwd=repo,
    check=True,
)
inference_process_seconds = perf_counter() - started
print(f"Inference process: {inference_process_seconds:.2f} s", flush=True)


## 5. Файл для бота и измерение времени

Одна JSON-запись на вопрос, только поле `code`. Переводы строк внутри программ
сохраняются как JSON-экранирование. Не копируйте программы из preview Kaggle:
скачайте сам файл `submission.jsonl` из Output. В Colab используйте панель Files
или вызовите `google.colab.files.download(str(submission))`.

Если замер превышает 180 секунд, это видно в метриках; наличие файла само по себе
не подтверждает соблюдение ограничения. Предыдущие прогоны инференса этой модели на T4 укладывались в 180 секунд;
здесь используется тот же код генерации, а новый замер сохраняется в метриках.

In [ ]:
predictions = [json.loads(line) for line in
               (output / "test/predictions.jsonl").read_text(encoding="utf-8").splitlines()]
if [row["sample_id"] for row in predictions] != list(range(1919)):
    raise ValueError("Нарушены число или порядок ответов")
if not all(isinstance(row["code"], str) for row in predictions):
    raise ValueError("Поле code должно быть строкой")

submission = WORK_DIR / "submission.jsonl"
with submission.open("w", encoding="utf-8") as file:
    for row in predictions:
        file.write(json.dumps({"code": row["code"]}, ensure_ascii=False) + "\n")

report = json.loads((output / "test/metrics.json").read_text(encoding="utf-8"))
if report["generated_examples"] != 1919 or report["num_beams"] != 1:
    raise ValueError("Настройки генерации не соответствуют сдаче")
if report["max_new_tokens"] != 1024:
    raise ValueError("Неожиданный лимит генерации")
report.update(
    checkpoint=checkpoint.name,
    inference_process_seconds=inference_process_seconds,
    within_generation_limit=report["generation_seconds"] <= 180,
    within_process_limit=inference_process_seconds <= 180,
)
metrics_file = WORK_DIR / "inference_metrics.json"
metrics_file.write_text(json.dumps(report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Submission:", submission, "rows:", len(predictions))
print("Metrics:", metrics_file)

from IPython.display import FileLink, display
display(FileLink(str(submission)))
display(FileLink(str(metrics_file)))
